# Step 5. Ethics and Bias Audit

This is where the dataset choice pays off. Step 5 opens the model up, states its limits, and checks whether it treats groups fairly.

I do four things.

1. Explain how the model decides, using SHAP.
2. State the model's limits honestly.
3. Audit the model for bias across the four sensitive attributes.
4. Apply two mitigations to one attribute, and show what they cost.

It loads the logistic regression chosen in Step 4. Being a simple, readable model, it makes the explanation clean.

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from src.paths import ROOT

model = joblib.load(ROOT / "models" / "model.joblib")
proc = ROOT / "data" / "processed"
train = pd.read_csv(proc / "train.csv")
test = pd.read_csv(proc / "test.csv")
ytr = train.pop("dropout")
yte = test.pop("dropout")
Xtr, Xte = train, test

y_pred = model.predict(Xte)
y_proba = model.predict_proba(Xte)[:, 1]
print("loaded model and split. test", Xte.shape)

## 1. How the Model Decides

A model that scores students should be able to say why. SHAP does this. SHAP gives each feature a value for each student, the push it gave to that student's risk score, up or down. Averaged over all students, it shows which features drive the model most.

SHAP fits a linear model cleanly, since each feature already has a weight. The chart below ranks features by their average push, largest first.

In [ ]:
import shap

pre = model.named_steps["pre"]
lr = model.named_steps["m"]
Ztr = pre.transform(Xtr)
Zte = pre.transform(Xte)
Ztr = Ztr.toarray() if hasattr(Ztr, "toarray") else Ztr
Zte = Zte.toarray() if hasattr(Zte, "toarray") else Zte
names = list(pre.get_feature_names_out())

explainer = shap.LinearExplainer(lr, shap.sample(Ztr, 100, random_state=42))
shap_values = explainer.shap_values(Zte)

mean_abs = np.abs(shap_values).mean(0)
order = np.argsort(mean_abs)[::-1][:12]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([names[i] for i in order][::-1], mean_abs[order][::-1])
ax.set_xlabel("mean absolute SHAP value")
ax.set_title("what drives the risk score")
fig.tight_layout()
fig.savefig(ROOT / "reports" / "figures" / "shap_importance.png", dpi=90)
plt.show()

The socioeconomic pressure feature drives the model most, ahead of tuition status. Both are money-strain signals, which fits what I found in Step 3. Course and gender come next, then admission grade and age. Gender showing up here matters, since it means the model uses gender directly, which the fairness audit below examines.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# PDP needs float columns, so I cast the two features I plot.
Xpdp = Xtr.copy()
for c in ["Age at enrollment", "socioeconomic pressure"]:
    Xpdp[c] = Xpdp[c].astype(float)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
PartialDependenceDisplay.from_estimator(model, Xpdp, ["Age at enrollment"], ax=ax[0])
PartialDependenceDisplay.from_estimator(model, Xpdp, ["socioeconomic pressure"], ax=ax[1])
fig.tight_layout()
fig.savefig(ROOT / "reports" / "figures" / "pdp.png", dpi=90)
plt.show()

Both curves rise. Predicted dropout risk climbs with age at enrollment, and it climbs with each added strain signal in the socioeconomic pressure score. Neither line is flat, so both features move the model in the direction Step 3 suggested.

## 2. The Limits

An honest audit names what could go wrong.

Imbalance. The dropout rate is 39 percent, mildly uneven. Class weights handled it, and PR AUC leads, which stays honest on uneven data. So imbalance is handled, not ignored.

Leakage. The biggest guard. The model trained on enrollment-time features only, with the twelve curricular columns dropped, since they record the outcome as it happens. Tuition status is a softer case, a near-outcome signal kept but watched, and Step 3 measured its effect.

Overfitting. Checked below, by comparing the model on the training set against the test set.

Scope. From Step 1, this model fits an IPP style Portuguese polytechnic, not any school anywhere. The results do not transfer to a very different institution.

In [ ]:
from sklearn.metrics import average_precision_score

train_pr = average_precision_score(ytr, model.predict_proba(Xtr)[:, 1])
test_pr = average_precision_score(yte, y_proba)
print("PR AUC on train", round(train_pr, 3))
print("PR AUC on test ", round(test_pr, 3))
print("gap", round(train_pr - test_pr, 3))

The gap between train and test PR AUC is small, 0.851 against 0.822. A small gap means the model learned the pattern, not the noise, so it is not badly overfit. The class weights and the simple linear model both help keep it steady.

## 3. The Fairness Audit

Step 2 showed large dropout-rate gaps across groups. A model trained on that data can carry the gaps into its flags. Here I measure that with three fairness metrics, checked across gender, scholarship, debtor, and age band.

Demographic parity difference. Do groups get flagged at the same rate? 0 is equal, larger is worse.

Equalized odds difference. Are the error rates, the catches and the false alarms, the same across groups? 0 is equal, larger is worse.

Disparate impact ratio. The flag rate of the lower group divided by the higher group. 1.0 is equal, and a common bar treats below 0.8 as a concern.

In [ ]:
from fairlearn.metrics import (
    MetricFrame, demographic_parity_difference, equalized_odds_difference,
    demographic_parity_ratio, selection_rate, true_positive_rate, false_positive_rate,
)

age_band = pd.cut(Xte["Age at enrollment"], [16, 20, 23, 30, 100],
                  labels=["17 to 20", "21 to 23", "24 to 30", "31 plus"])
attrs = {
    "Gender": Xte["Gender"].map({1: "male", 0: "female"}),
    "Scholarship": Xte["Scholarship holder"].map({1: "holder", 0: "none"}),
    "Debtor": Xte["Debtor"].map({1: "debtor", 0: "not debtor"}),
    "Age band": age_band,
}

rows = []
for name, sf in attrs.items():
    rows.append({
        "attribute": name,
        "DP diff": round(demographic_parity_difference(yte, y_pred, sensitive_features=sf), 3),
        "EO diff": round(equalized_odds_difference(yte, y_pred, sensitive_features=sf), 3),
        "DI ratio": round(demographic_parity_ratio(yte, y_pred, sensitive_features=sf), 3),
    })
audit = pd.DataFrame(rows)
print(audit.to_string(index=False))

print()
print("gender, per group")
mf = MetricFrame(
    metrics={"flag rate": selection_rate, "recall": true_positive_rate, "false alarm": false_positive_rate},
    y_true=yte, y_pred=y_pred, sensitive_features=attrs["Gender"],
)
print(mf.by_group.round(3))

Every attribute fails. The disparate impact ratio sits well below 0.8 for all four, and the differences are large. Age band is the worst, then debtor and scholarship, then gender.

The per-group view for gender shows the shape of it. The model flags men at 65 percent and women at 32. It catches more male dropouts, recall 0.84 against 0.69, but it also raises far more false alarms for men, 0.43 against 0.16. So the model leans hard on flagging men. This is what I try to reduce next.

## 4. Mitigation

All four attributes cannot be fixed at once, since pushing one into balance can pull another out. So one attribute gets fixed in full, gender, and the rest audited. Gender is a clear protected attribute with a large gap, which makes it the right first target.

Two different methods follow.

Threshold optimizer. A post-processing method. It keeps the model as is and picks a separate cutoff for each group, aiming to equalize the error rates.

Exponentiated gradient. An in-processing method. It retrains the model under a fairness constraint, here aiming to equalize the flag rate across groups.

Both are compared against the baseline on fairness and on accuracy, so the trade-off is visible.

In [ ]:
from sklearn.metrics import recall_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

g_tr = Xtr["Gender"].map({1: "male", 0: "female"})
g_te = attrs["Gender"]

def fairness_row(name, pred):
    return {
        "approach": name,
        "recall": round(recall_score(yte, pred), 3),
        "accuracy": round(accuracy_score(yte, pred), 3),
        "DP diff": round(demographic_parity_difference(yte, pred, sensitive_features=g_te), 3),
        "EO diff": round(equalized_odds_difference(yte, pred, sensitive_features=g_te), 3),
        "DI ratio": round(demographic_parity_ratio(yte, pred, sensitive_features=g_te), 3),
    }

rows = [fairness_row("baseline", y_pred)]

# post-processing, aim for equal error rates
to = ThresholdOptimizer(estimator=model, constraints="equalized_odds",
                        predict_method="predict_proba", prefit=True)
to.fit(Xtr, ytr, sensitive_features=g_tr)
rows.append(fairness_row("threshold opt, equal odds",
                         to.predict(Xte, sensitive_features=g_te, random_state=42)))

# in-processing, aim for equal flag rates
eg = ExponentiatedGradient(LogisticRegression(max_iter=2000, class_weight="balanced"),
                           constraints=DemographicParity())
eg.fit(Ztr, ytr, sensitive_features=g_tr)
rows.append(fairness_row("exp. gradient, equal flag rate", eg.predict(Zte, random_state=42)))

mitigation = pd.DataFrame(rows)
print(mitigation.to_string(index=False))

Both methods cut the unfairness, in different ways, and both cost something.

The threshold optimizer nearly erases the error-rate gap. Equalized odds difference falls from 0.277 to 0.011. But it flags fewer students overall, so recall drops from 0.775 to 0.602. It buys equal error rates with a real loss of catches.

The exponentiated gradient nearly erases the flag-rate gap. Demographic parity difference falls from 0.338 to 0.036, and the disparate impact ratio rises to 0.919, above the 0.8 bar. It costs less recall, 0.711, since it retrains rather than just reweighting cutoffs.

Neither is free. The right choice depends on which fairness definition matters most, equal error rates or equal flag rates. For an early-warning tool, where catching students is the point, the exponentiated gradient reads better here, since it holds more recall.

## 5. Residual Risk

One attribute is fixed, not the problem.

Gender is balanced, but scholarship, debtor, and age band still fail. Fixing one can worsen another, so a full fix would weigh them together, which is harder and beyond this pass.

Every mitigation costs recall, and recall is the metric that matters most, from Step 1. So fairness and catching at-risk students pull against each other here. That tension is real and worth stating, not hiding.

The features that carry the bias, being a debtor, owing tuition, holding no scholarship, are also the true strain signals that make the model work. Removing them would make the model fairer and weaker at the same time. The honest position is that this model needs a human in the loop, not blind trust, and its flags should start a conversation, not a verdict.

## What Step 5 Settles

1. SHAP shows the model leans on socioeconomic pressure and tuition status, with gender in the mix.
2. The limits are named, mild imbalance handled, leakage guarded, small overfitting gap, single-institution scope.
3. The fairness audit finds real bias across all four sensitive attributes.
4. Two mitigations reduce the gender gap, each with a clear cost in recall.
5. The residual risk is stated plainly, so the model is used with care, not blind trust.

This closes the core seven steps. The bonus work, deployment and a GenAI explanation layer, builds on the model saved here.